# Phase 4: 동적 SL 에이전트 학습
144만 건 "방향 맞지만 SL 히트" 문제를 해결하기 위한 PPO 기반 동적 SL 에이전트.

## 0. 패키지 설치 (최초 1회)

In [ ]:
# 최초 1회만 실행
# !pip install stable-baselines3 gymnasium pandas numpy

## 1. 경로 설정

In [ ]:
import sys
from pathlib import Path

# ★ 프로젝트 루트 경로 — 본인 환경에 맞게 수정
PROJECT_ROOT = Path(r"D:/AutoTrade")

# sys.path에 프로젝트 루트 추가 (rl 패키지 import용)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 데이터 경로
SIGNALS_CSV = PROJECT_ROOT / "Raw_Data" / "labeled_signals" / "signals_all_labeled.csv"
OHLCV_DIR = PROJECT_ROOT / "Raw_Data" / "CRYPTO_BINANCE_15M"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "sl_agent"

print(f"시그널 CSV: {SIGNALS_CSV}  (존재: {SIGNALS_CSV.exists()})")
print(f"OHLCV 디렉토리: {OHLCV_DIR}  (존재: {OHLCV_DIR.exists()})")

## 2. 데이터 확인

In [ ]:
import pandas as pd

df = pd.read_csv(SIGNALS_CSV, nrows=5)
print(f"컬럼 수: {len(df.columns)}")
print(f"컬럼: {list(df.columns)}")
df.head()

In [ ]:
# 전체 데이터 기본 통계
df_full = pd.read_csv(SIGNALS_CSV)
print(f"총 시그널: {len(df_full):,}건")
print(f"\nexit_type 분포:")
print(df_full["exit_type"].value_counts())
print(f"\n방향 정확도: {df_full['direction_correct'].mean():.1%}")
print(f"SL 히트 중 방향 맞음: {df_full[df_full['exit_type']=='sl']['direction_correct'].mean():.1%}")

## 3. 관측 벡터 빌드 (소량 테스트)

In [ ]:
from rl.features.state_builder import build_from_csv

# ★ 먼저 소량(1000건)으로 데이터 파이프라인 검증
test_data = build_from_csv(
    signals_csv=SIGNALS_CSV,
    ohlcv_dir=OHLCV_DIR,
    max_signals=1000,
)

print(f"관측 벡터: {test_data['observations'].shape}")
print(f"SL 거리 범위: {test_data['base_sl_distances'].min():.4f} ~ {test_data['base_sl_distances'].max():.4f}")
print(f"방향 분포: LONG {(test_data['directions']==1).sum()}, SHORT {(test_data['directions']==-1).sum()}")

## 4. 환경 동작 테스트

In [ ]:
from rl.env.sl_env import SLAdjustEnv

env = SLAdjustEnv(
    signals=test_data["observations"],
    ohlc_slices=test_data["ohlc_slices"],
    base_sl_distances=test_data["base_sl_distances"],
    tp_distances=test_data["tp_distances"],
    entry_prices=test_data["entry_prices"],
    directions=test_data["directions"],
    shuffle=False,
)

# 수동으로 몇 스텝 테스트
obs, info = env.reset()
print(f"관측 shape: {obs.shape}")
print(f"관측 값: {obs}")

# 배수 1.0 (기본 SL) 테스트
action_1x = [0.0]  # [-1,1] → [0.5,3.0], 0.0 → 1.75
action_baseline = [-0.6]  # → ~1.0 (기본 SL)

obs2, reward, terminated, truncated, info = env.step(action_baseline)
print(f"\n기본 SL 결과: reward={reward:.4f}, exit={info['exit_type']}, ROE={info['roe']:.4f}")

## 5. 학습 (소량 빠른 테스트)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

from rl.agent.sl_agent import SLAgent, TrainConfig

# ★ 먼저 소량 + 적은 스텝으로 파이프라인 검증
config = TrainConfig(
    total_timesteps=5_000,      # 테스트용 (본학습: 500_000)
    n_steps=512,                # 테스트용 (본학습: 2048)
    batch_size=128,
    output_dir=str(OUTPUT_DIR / "test_run"),
)

agent = SLAgent(config=config)
results = agent.train(
    signals_csv=SIGNALS_CSV,
    ohlcv_dir=OHLCV_DIR,
    config=config,
    max_signals=2000,  # 테스트용 소량
)

In [ ]:
# 결과 확인
for wname, w_results in results.items():
    for r in w_results:
        print(
            f"{r.window_name}/{r.split}: "
            f"ROE {r.mean_roe:+.4f} (기준 {r.mean_roe_baseline:+.4f}, "
            f"개선 {r.roe_improvement:+.4f}) "
            f"PF {r.profit_factor:.2f} (기준 {r.profit_factor_baseline:.2f}) "
            f"평균배수 {r.mean_sl_mult:.2f}"
        )

## 6. 본학습 (전체 데이터)
소량 테스트가 정상 동작하면 아래 셀 실행. CPU 24코어 기준 수십 분 소요.

In [ ]:
# ★ 본학습 — 테스트 통과 후 실행
config_full = TrainConfig(
    total_timesteps=500_000,
    n_steps=2048,
    batch_size=256,
    learning_rate=3e-4,
    output_dir=str(OUTPUT_DIR),
)

agent_full = SLAgent(config=config_full)
results_full = agent_full.train(
    signals_csv=SIGNALS_CSV,
    ohlcv_dir=OHLCV_DIR,
    config=config_full,
    # max_signals=None,  # 전체 데이터
)

## 7. 학습된 모델 로드 & 추론

In [ ]:
# 저장된 모델 로드
best_model_path = OUTPUT_DIR / "Window3" / "best" / "best_model.zip"
agent_loaded = SLAgent.load(best_model_path)

# 단일 시그널 추론 예시
import numpy as np
sample_obs = test_data["observations"][0]
sl_mult = agent_loaded.predict(sample_obs)
print(f"SL 배수: {sl_mult:.2f}x (기본 SL의 {sl_mult:.1f}배)")

# 배치 추론
mults = agent_loaded.predict_batch(test_data["observations"][:100])
print(f"\n배치 추론 (100건):")
print(f"  평균 배수: {mults.mean():.2f}")
print(f"  범위: {mults.min():.2f} ~ {mults.max():.2f}")

## 8. 프로덕션 적용
`auto_trader_v9.py`에서 사용할 때:

In [ ]:
# auto_trader_v9.py 연동 예시 (실제 적용 시)
"""
from rl.agent.sl_agent import SLAgent

# 봇 시작 시 모델 로드
sl_agent = SLAgent.load("experiments/sl_agent/Window3/best/best_model.zip")

# 시그널 발생 시
def calc_dynamic_sl(signal, market_state):
    obs = build_observation(signal, market_state)  # 13차원 벡터
    sl_mult = sl_agent.predict(obs)
    
    # 기존 고정 SL에 배수 적용
    if signal.direction == 'LONG':
        dynamic_sl = signal.squeeze_low - signal.buffer * sl_mult
    else:
        dynamic_sl = signal.squeeze_high + signal.buffer * sl_mult
    
    return dynamic_sl
"""
print("프로덕션 연동 코드 예시 — 위 docstring 참고")